# Demo: end-to-end inference with the trained GNN-BERT fusion model

Loads one track's cached structure graph + caption, runs it through the
**Task 3** fusion checkpoint (`results/checkpoints/fusion_cross_attn.pt`),
and prints the predicted tags and predicted valence/arousal.

Prerequisites (run once before this notebook):
```bash
python src/audio_features.py
python src/graph_builder.py
python src/train.py --task fusion --fusion_mode cross_attn
```

In [3]:
import sys, os, json
sys.path.append(os.path.abspath("../src"))

import torch
from torch_geometric.data import Batch

from utils import load_config, get_device
from datasets import load_dataset
from bert_encoder import get_tokenizer, tokenize_batch
from fusion_model import GNNBertFusion

os.chdir("..")  # so config.yaml's relative paths resolve from the repo root
cfg = load_config("config.yaml")
device = get_device(cfg["train"]["device"])
print("device:", device)

device: cuda


## 1. Pick a track and load its cached graph + caption

Swap `TRACK_ID` for any track that has both a graph in `data/processed/graphs/`
and a caption in the MusicCaps CSV (or an empty string is fine, too).

In [6]:
TRACK_ID = "j5R0hwX27Gk"  # e.g. an ytid from musiccaps-public.csv

graph_path = os.path.join(cfg["paths"]["processed_dir"], "graphs", f"{TRACK_ID}.pt")
graph = torch.load(graph_path, weights_only=False)
batch = Batch.from_data_list([graph]).to(device)

captions = load_dataset(cfg["paths"]["raw_dir"])
caption_text = captions.get(TRACK_ID, "")
print("Caption:", caption_text or "(none found — using empty string)")

tokenizer = get_tokenizer(cfg["text"]["bert_model_name"])
tok = tokenize_batch(tokenizer, [caption_text], cfg["text"]["max_length"])
input_ids = tok["input_ids"].to(device)
attention_mask = tok["attention_mask"].to(device)

Resolving data files:   0%|          | 0/35581 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

DatasetGenerationError: An error occurred while generating the dataset

## 2. Load the trained fusion checkpoint

In [ ]:
tag_table = load_tag_table(cfg["paths"]["raw_dir"], cfg["datasets"]["active_audio"], cfg["model"]["num_tags"])
tag_names = list(tag_table.columns)

model = GNNBertFusion(audio_in_dim=cfg["audio"]["n_mels"], cfg=cfg, fusion_mode="cross_attn").to(device)
ckpt_path = os.path.join(cfg["logging"]["checkpoint_dir"], "fusion_cross_attn.pt")
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()
print(f"Loaded checkpoint from {ckpt_path}")

## 3. Run inference and inspect the prediction

In [ ]:
with torch.no_grad():
    tag_logits, emotion_pred = model(batch.x, batch.edge_index, batch.batch, input_ids, attention_mask)
    tag_probs = torch.sigmoid(tag_logits)[0].cpu().numpy()
    valence, arousal = emotion_pred[0].cpu().numpy()

top5_idx = tag_probs.argsort()[::-1][:5]
print(f"Track: {TRACK_ID}")
print("\nTop-5 predicted tags:")
for i in top5_idx:
    print(f"  {tag_names[i]:<20s} {tag_probs[i]:.3f}")
print(f"\nPredicted valence: {valence:+.3f}  (range -1 to 1)")
print(f"Predicted arousal: {arousal:+.3f}  (range -1 to 1)")